# Stage 13B — GLM balancing stress screen

Compare the frozen uniform Sinkhorn rule with no balancing on the same rows within each fold. All probabilities come from the existing prompt-held-out GLM cache. These are development stress scenarios; they do not estimate hidden-test performance or change S07.


## Private CPU input

Bind only the private Stage 13 GLM readout V3 output (kernel version **368292220**). Keep the notebook private, with accelerator None and Internet off. The runner verifies the frozen cache before scoring. Row-level predictions and identifiers stay inside the private input.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import subprocess
import sys

_runner_source = '"""Run one predeclared balancing diagnostic on the private GLM OOF cache."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\n\nCOMPLETE = "COMPLETE_GLM_SINKHORN_STRESS_ONLY"\nSCOPE = "glm_only_paired_fold_local_balancing_stress"\nSOURCE_VERSION = 368292220\nCACHE_SHA = "565df6f8e652a9850ff11cf4e8b6a81fcccbc7db593186ff7f53358e7190bd16"\nPROB_SHA = "41d80bc9350ab44d5aabe02b24e0174881a307bb1eb962f04196682e7faa94fe"\nID_SHA = "707edf645b0f09bba8f0983b5bc9f522595c95e8ca5b763b38babeb38200368d"\nLABEL_SHA = "088b14188775e41a314f269da3af74f96144a56caa242ccbeacc9e3c817a7e08"\nFOLD_SHA = "a55b035686cb7b31f6fbe852ac990668709aec4b913c5d0928d89d843bdbfe62"\nEXPECTED_RECALL = 61547 / 134100\nK, ROWS, PROMPTS, FOLDS = 18, 2699, 150, 5\nSCENARIOS = ("natural_full", "matched_balanced", "skew_a", "skew_b")\nDRAWS, SEED = 5000, 20261111\nATOL = 1e-12\n\n\nclass GateError(Exception):\n    pass\n\n\ndef require(condition, code):\n    if not bool(condition):\n        raise GateError(code)\n\n\ndef digest_json(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef salted_hash(salt, value):\n    return hashlib.sha256((salt + "\\0" + str(value)).encode("utf-8")).digest()\n\n\ndef one(root, name):\n    matches = [p for p in root.rglob(name) if p.is_file()]\n    require(len(matches) == 1, "unique_" + name.replace(".", "_"))\n    return matches[0]\n\n\ndef validate_p(p):\n    require(p.ndim == 2 and p.shape[1] == K and p.dtype == np.float64 and len(p) > 0,\n            "probability_shape_or_dtype")\n    require(np.isfinite(p).all() and (p > 0).all() and (p <= 1).all(), "invalid_probability_values")\n    require(np.max(np.abs(p.sum(axis=1) - 1.0)) <= ATOL, "probability_row_sums")\n\n\ndef evidence(p, uniform):\n    """Use the source\'s fixed 100 updates and branch threshold on existing P."""\n    validate_p(p)\n    if not uniform or len(p) < K:\n        return np.log(p + 1e-12), None\n    q = p.copy()\n    with np.errstate(divide="raise", invalid="raise", over="raise"):\n        for _ in range(100):\n            q *= (len(q) / 18) / q.sum(0, keepdims=True)\n            q /= q.sum(1, keepdims=True)\n    validate_p(q)\n    residual = float(np.max(np.abs(q.sum(axis=0) - len(q) / K)))\n    require(np.isfinite(residual), "nonfinite_column_mass_error")\n    return np.log(q + 1e-12), residual\n\n\ndef cache_inputs(root):\n    cache = one(root, "heldout_probabilities.npz")\n    report = one(root, "readout_report.json")\n    require(cache.parent == report.parent, "cache_report_parent_mismatch")\n    with cache.open("rb") as f:\n        require(hashlib.file_digest(f, "sha256").hexdigest() == CACHE_SHA, "cache_byte_hash_mismatch")\n    r = json.loads(report.read_text(encoding="utf-8"))\n    require(r.get("status") == "COMPLETE_GLM_PLAIN_GROUPED_DEVELOPMENT_REFERENCE"\n            and r.get("scope") == "glm_only_prompt_heldout_development"\n            and r.get("heldout_cache_sha256") == CACHE_SHA\n            and r.get("probability_values_sha256") == PROB_SHA\n            and r.get("admission_checks_passed") == 7, "producer_report_contract")\n    for flag in ("fit_performed", "heldout_predictions_produced", "heldout_cache_readback_passed"):\n        require(r.get(flag) is True, "producer_report_missing_required_flag")\n    for flag in ("balancing_performed", "official_score_estimated", "full_s07_oof_admitted", "raw_rows_or_predictions_exported"):\n        require(r.get(flag) is False, "producer_report_scope")\n    keys = ("ids", "prompt_ids", "targets", "folds", "classes", "probabilities")\n    with np.load(cache, allow_pickle=False) as z:\n        require(len(z.files) == len(keys) and set(z.files) == set(keys), "cache_schema")\n        ids, prompts, targets, folds, classes, p = (z[k] for k in keys)\n    require(ids.shape == prompts.shape == targets.shape == folds.shape == (ROWS,), "cache_row_axes")\n    require(ids.dtype.kind == prompts.dtype.kind == classes.dtype.kind == "U"\n            and targets.dtype == folds.dtype == np.int64 and classes.shape == (K,), "cache_axis_dtypes")\n    require(len(np.unique(ids)) == ROWS and len(np.unique(prompts)) == PROMPTS, "cache_unique_axes")\n    require(digest_json(ids.tolist()) == ID_SHA and digest_json(classes.tolist()) == LABEL_SHA, "cache_axis_hashes")\n    require(np.array_equal(np.unique(targets), np.arange(K)) and np.array_equal(np.unique(folds), np.arange(1, 6)),\n            "cache_class_or_fold_indices")\n    require(np.bincount(targets).tolist() == [149] + [150] * 17, "original_class_supports")\n    validate_p(p)\n    require(len(p) == ROWS and hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == PROB_SHA,\n            "cache_probability_hash")\n    mapping = {}\n    for group, fold in zip(prompts, folds):\n        require(mapping.setdefault(str(group), int(fold)) == int(fold), "prompt_crosses_fold")\n    require(digest_json(mapping) == FOLD_SHA and r.get("canonical_prompt_fold_mapping_sha256") == FOLD_SHA,\n            "canonical_fold_map_hash")\n    require(r.get("classes") == classes.tolist() and len(r.get("folds", [])) == FOLDS, "producer_fold_axis")\n    for f, item in enumerate(r["folds"], 1):\n        n = 540 if f < 5 else 539\n        mask = folds == f\n        require(item.get("fold") == f and int(mask.sum()) == n and len(np.unique(prompts[mask])) == 30\n                and [item.get(k) for k in ("fit_rows", "valid_rows", "fit_prompts", "valid_prompts", "prompt_overlap")]\n                == [ROWS - n, n, 120, 30, 0], "producer_fold_exclusion_contract")\n    return ids, prompts, targets, folds, classes, p, r\n\n\ndef mask_indices(ids, targets, classes, indices):\n    """Rank once per class. Use matched subsets for all paired scoring arms."""\n    ranked = sorted(range(K), key=lambda c: (salted_hash("stage13b-class-v1", classes[c]), str(classes[c]).encode("utf-8")))\n    a_major = set(ranked[:9])\n    buckets = [sorted(indices[targets[indices] == c].tolist(),\n                      key=lambda i: (salted_hash("stage13b-row-v1", ids[i]), str(ids[i]).encode("utf-8"))) for c in range(K)]\n    require(all(len(rows) >= 27 for rows in buckets), "insufficient_mask_class_support")\n    masks = {"natural_full": indices}\n    for name in SCENARIOS[1:]:\n        counts = [20] * K if name == "matched_balanced" else [\n            27 if ((c in a_major) == (name == "skew_a")) else 13 for c in range(K)]\n        chosen = np.asarray(sorted(i for c, rows in enumerate(buckets) for i in rows[:counts[c]]), dtype=np.int64)\n        require(len(chosen) == len(np.unique(chosen)) == 360 and np.bincount(targets[chosen], minlength=K).tolist() == counts,\n                "matched_mask_supports")\n        masks[name] = chosen\n    return masks\n\n\ndef confusion(y, pred):\n    return np.bincount(y * K + pred, minlength=K*K).reshape(K, K)\n\n\ndef recalls(counts):\n    totals = counts.sum(axis=1)\n    require((totals > 0).all(), "class_missing_from_score")\n    return counts.diagonal() / totals\n\n\ndef bootstrap_delta(prompt_index, targets, correct_off, correct_uniform, weights):\n    supports = np.zeros((30, K), dtype=np.int64)\n    differences = np.zeros_like(supports)\n    np.add.at(supports, (prompt_index, targets), 1)\n    np.add.at(differences, (prompt_index, targets), correct_off.astype(np.int64) - correct_uniform.astype(np.int64))\n    return weights @ supports, weights @ differences\n\n\ndef diagnostics(ids, prompts, targets, folds, classes, p, producer):\n    # Freeze the shared paired resampling before applying either arm.\n    rng = np.random.default_rng(SEED)\n    weights = [rng.multinomial(30, np.full(30, 1/30), size=DRAWS) for _ in range(FOLDS)]\n    stores = {n: {"off": np.zeros((K, K), dtype=np.int64), "uniform": np.zeros((K, K), dtype=np.int64),\n                  "support_draws": np.zeros((DRAWS, K), dtype=np.int64), "delta_draws": np.zeros((DRAWS, K), dtype=np.int64),\n                  "fixed": 0, "broken": 0, "changed": 0, "batches": []} for n in SCENARIOS}\n    digest_before = hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest()\n    for f in range(1, 6):\n        indices = np.flatnonzero(folds == f)\n        ordered_groups = np.unique(prompts[indices])\n        require(len(ordered_groups) == 30, "scoring_batch_prompt_count")\n        for name, selected in mask_indices(ids, targets, classes, indices).items():\n            require(len(np.unique(prompts[selected])) == 30 and (folds[selected] == f).all(), "masked_batch_fold_or_prompt_count")\n            y, pp = targets[selected], p[selected]\n            log_off, _ = evidence(pp, False)\n            log_uniform, residual = evidence(pp, True)\n            pred_off, pred_uniform = log_off.argmax(1), log_uniform.argmax(1)\n            off, uniform = confusion(y, pred_off), confusion(y, pred_uniform)\n            co, cu = pred_off == y, pred_uniform == y\n            fixed, broken = int((co & ~cu).sum()), int((cu & ~co).sum())\n            changed = int((pred_off != pred_uniform).sum())\n            require(fixed + broken <= changed and int(co.sum() - cu.sum()) == fixed - broken, "paired_counts")\n            b = stores[name]\n            b["off"] += off\n            b["uniform"] += uniform\n            b["fixed"] += fixed\n            b["broken"] += broken\n            b["changed"] += changed\n            b["batches"].append(dict(fold=f, rows=len(y), prompts=30, class_supports=off.sum(1).tolist(),\n                                     off_ba=float(recalls(off).mean()), uniform_ba=float(recalls(uniform).mean()),\n                                     uniform_max_column_mass_error=residual))\n            ds, dd = bootstrap_delta(np.searchsorted(ordered_groups, prompts[selected]), y, co, cu, weights[f-1])\n            b["support_draws"] += ds\n            b["delta_draws"] += dd\n    natural = stores["natural_full"]["off"]\n    require(np.array_equal(natural, np.asarray(producer["confusion_counts"]))\n            and abs(float(recalls(natural).mean()) - EXPECTED_RECALL) <= 1e-15\n            and abs(float(producer["pooled_balanced_accuracy"]) - EXPECTED_RECALL) <= 1e-15, "original_plain_reference_replay")\n    require(hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == digest_before, "mutated_upstream_probabilities")\n    results = {}\n    for name, b in stores.items():\n        off, uniform = b["off"], b["uniform"]\n        ro, ru = recalls(off), recalls(uniform)\n        require(np.array_equal(off.sum(1), uniform.sum(1)) and (b["support_draws"] > 0).all(), "paired_support_or_bootstrap_coverage")\n        intervals = np.percentile((b["delta_draws"] / b["support_draws"]).mean(axis=1) * 100, [2.5, 97.5])\n        results[name] = dict(rows=int(off.sum()), batches=b["batches"], off_ba=float(ro.mean()), uniform_ba=float(ru.mean()),\n                             delta_pp=float((ro.mean() - ru.mean()) * 100), conditional_delta_ci95_pp=intervals.tolist(),\n                             class_supports=off.sum(1).tolist(), class_delta_pp=((ro - ru) * 100).tolist(),\n                             off_class_recalls=ro.tolist(), uniform_class_recalls=ru.tolist(),\n                             off_confusion_counts=off.tolist(), uniform_confusion_counts=uniform.tolist(),\n                             corrected_by_off=b["fixed"], broken_by_off=b["broken"], changed_by_off=b["changed"])\n    require([results[n]["rows"] for n in SCENARIOS] == [2699, 1800, 1800, 1800], "scenario_row_counts")\n    criteria = dict(\n        both_skew_point_gains_ge_half_pp=all(results[n]["delta_pp"] >= 0.5 for n in ("skew_a", "skew_b")),\n        both_skew_conditional_ci_lowers_positive=all(results[n]["conditional_delta_ci95_pp"][0] > 0 for n in ("skew_a", "skew_b")),\n        natural_and_matched_losses_le_half_pp=all(results[n]["delta_pp"] >= -0.5 for n in ("natural_full", "matched_balanced")),\n        every_class_recall_loss_le_five_pp=all(min(b["class_delta_pp"]) >= -5.0 - 1e-12 for b in results.values()),\n    )\n    return results, criteria\n\n\ndef run(input_root, output_root):\n    t = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status="FAILED_GLM_SINKHORN_STRESS", scope=SCOPE, source_version_id=SOURCE_VERSION,\n                  heldout_cache_sha256=CACHE_SHA, fit_performed=False, official_score_estimated=False,\n                  full_s07_oof_admitted=False, raw_rows_or_predictions_exported=False)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        ids, groups, targets, folds, classes, p, prior = cache_inputs(root)\n        results, criteria = diagnostics(ids, groups, targets, folds, classes, p, prior)\n        report.update(status=COMPLETE, scenarios=results, classes=classes.tolist(), decision_criteria=criteria,\n                      passes_proxy_screen=all(criteria.values()), probabilities_unchanged=True,\n                      original_plain_reference_replayed=True, actual_folds=5, prompts=150,\n                      bootstrap=dict(draws=DRAWS, seed=SEED, scope="fixed-prediction stratified prompt resampling"),\n                      control=dict(alpha=1, iterations=100, minimum_batch_rows=18), candidate=dict(alpha=0),\n                      delta_direction="off_minus_uniform", numpy_version=np.__version__)\n    except GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - t, 6)\n    if writable:\n        try:\n            tmp = out / ".balancing_stress_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "balancing_stress_report.json")\n        except Exception as error:\n            report.update(status="FAILED_GLM_SINKHORN_STRESS", failure_code="report_write_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13b_balancing_stress")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13B_GLM_STRESS_JSON_BEGIN", flush=True)\n    print(json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13B_GLM_STRESS_JSON_END", flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
_runner_sha256 = 'f4981f385074f60ee034b979d4aa6672c7d28eaa408c374d5caf55a7c446fbd9'
_source_dir = Path('/kaggle/working/_stage13b_balancing_stress_source')
_runner_path = _source_dir / 'stage13b_balancing_stress.py'
_runner_bytes = _runner_source.encode('utf-8')
if hashlib.sha256(_runner_bytes).hexdigest() != _runner_sha256:
    raise RuntimeError('Stage13B embedded runner hash mismatch')
_source_dir.mkdir(parents=True, exist_ok=True)
_runner_path.write_bytes(_runner_bytes)
if hashlib.sha256(_runner_path.read_bytes()).hexdigest() != _runner_sha256:
    raise RuntimeError('Stage13B written runner hash mismatch')
print('STAGE13B_SOURCE_CHECKS_PASS', _runner_sha256)


## Run the frozen stress comparison

Scoring stops if any cache, fold, arithmetic, or scenario check fails. No model is fitted.


In [ ]:
subprocess.run([sys.executable, str(_runner_path), '--input-root', '/kaggle/input', '--output-root', '/kaggle/working/stage13b_balancing_stress'], check=True, timeout=180)


## Verify the saved aggregate report

Check the report scope, all four scenario row counts, and five batches per scenario. Display only the aggregate differences and the proxy-screen decision.


In [ ]:
_result_dir = Path('/kaggle/working/stage13b_balancing_stress')
_report = json.loads((_result_dir / 'balancing_stress_report.json').read_text(encoding='utf-8'))
def _require(condition, code):
    if not condition:
        raise RuntimeError('Stage13B saved report check failed: ' + code)
_require(type(_report) is dict, 'report_type')
_require(_report.get('status') == 'COMPLETE_GLM_SINKHORN_STRESS_ONLY', 'status')
_require(_report.get('scope') == 'glm_only_paired_fold_local_balancing_stress', 'scope')
_require(_report.get('source_version_id') == 368292220, 'source_version_id')
_require(_report.get('heldout_cache_sha256') == '565df6f8e652a9850ff11cf4e8b6a81fcccbc7db593186ff7f53358e7190bd16', 'cache_sha256')
_require(_report.get('probabilities_unchanged') is True, 'probabilities_unchanged')
for _key in ['fit_performed', 'official_score_estimated', 'full_s07_oof_admitted', 'raw_rows_or_predictions_exported']:
    _require(_report.get(_key) is False, _key)
_require(type(_report.get('passes_proxy_screen')) is bool, 'passes_proxy_screen')
_expected_rows = {'natural_full': 2699, 'matched_balanced': 1800, 'skew_a': 1800, 'skew_b': 1800}
_scenarios = _report.get('scenarios')
_require(type(_scenarios) is dict and set(_scenarios) == set(_expected_rows), 'scenario_names')
_delta_pp = {}
for _name, _rows in _expected_rows.items():
    _scenario = _scenarios[_name]
    _require(type(_scenario) is dict and type(_scenario.get('rows')) is int and _scenario['rows'] == _rows, 'scenario_rows_' + _name)
    _batches = _scenario.get('batches')
    _require(type(_batches) is list and len(_batches) == 5, 'batch_count_' + _name)
    _delta = _scenario.get('delta_pp')
    _require(type(_delta) in (int, float) and math.isfinite(_delta) and -100 <= _delta <= 100, 'delta_pp_' + _name)
    _delta_pp[_name] = _delta
print('STAGE13B_SAVED_REPORT_READBACK_PASS', json.dumps({'scenario_delta_pp': _delta_pp, 'passes_proxy_screen': _report['passes_proxy_screen']}, sort_keys=True))
